# 623 SPP candidate gate: stateful LSTM vs sliding causal CNN

This notebook trains only the independent `offline_spp` candidate-gating track. SPP's private ST/PT/GHR/confidence/filter state generates a fixed causal action bank; the NN sees only restricted address-derived and candidate address/rank inputs and can suppress but not invent actions. Captured fill level is replay metadata, never a model feature.

The CNN follows the professor sketch exactly: one causal `Conv1d` layer, a 3-event filter window, stride 1, dilation 1, and two rows of left-only padding. Output at event t sees only `[t-2, t-1, t]`. It is a shallow sliding CNN, not the previous six-layer dilated TCN.


In [ ]:
import hashlib, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(torch.cuda.get_device_name(0), torch.__version__)

REPO = '/content/cache_arch'
PUBLIC_URL = 'https://github.com/Angelawoo572/cache_arch.git'
TOKEN = userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN in Colab Secrets'
ASKPASS = '/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *) echo "$GITHUB_TOKEN" ;;\nesac\n')
os.chmod(ASKPASS, 0o700)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': ASKPASS, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': TOKEN})
try:
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', PUBLIC_URL, REPO], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', 'main'], check=True, env=git_env)
finally:
    pathlib.Path(ASKPASS).unlink(missing_ok=True)
print('Git:', subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ID = '623_offline_lstm_cnn_spp_seed7'
DRIVE_ROOT = f'/content/drive/MyDrive/cache_prefetch_623_spp_cnn/{RUN_ID}'
INPUT_DIR = f'{DRIVE_ROOT}/colab_input'
OUTPUT_ROOT = f'{DRIVE_ROOT}/colab_output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
INPUT_ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_input.tar.gz'
REUPLOAD_INPUT = True  # fail-safe: do not silently reuse a stale archive
if REUPLOAD_INPUT or not os.path.isfile(INPUT_ARCHIVE):
    from google.colab import files
    expected = f'{RUN_ID}.colab_input.tar.gz'
    uploaded = files.upload()
    assert expected in uploaded, f'Select {expected}; got {list(uploaded)}'
    pathlib.Path(INPUT_ARCHIVE).write_bytes(uploaded[expected])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR, exist_ok=True)
with tarfile.open(INPUT_ARCHIVE, 'r:gz') as archive:
    archive.extractall(INPUT_DIR)
checksums = pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines()
for record in checksums:
    expected_sha, filename = record.split(maxsplit=1)
    filename = filename.lstrip('*')
    observed_sha = hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{filename}').read_bytes()).hexdigest()
    assert observed_sha == expected_sha, f'archive SHA256 mismatch: {filename}'
print('Input:', INPUT_ARCHIVE)
print('Persistent output:', OUTPUT_ROOT)


In [ ]:
import gzip, hashlib, json
TRACE = '623.xalancbmk_s-700B'
POLICY = 'spp'
ROLES = ('train', 'guard', 'eval')
INPUTS = {}
for role in ROLES:
    stream = f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz'
    candidates = f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_candidates.csv.gz'
    INPUTS[role] = {'stream': stream, 'candidates': candidates}
    for path in (stream, candidates):
        assert os.path.isfile(path), path
        digest = hashlib.sha256()
        with gzip.open(path, 'rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        print(path, digest.hexdigest())
COLLECTION_MANIFEST = f'{INPUT_DIR}/collection_manifest.json'
assert os.path.isfile(COLLECTION_MANIFEST), COLLECTION_MANIFEST
collection = json.loads(pathlib.Path(COLLECTION_MANIFEST).read_text())
assert collection['status'] == 'PASS'
assert collection['experiment_revision'] == 'spp_gate_sliding_cnn_v1'
assert collection['event_logger_schema'] == '623_causal_trigger_v5'
assert collection['candidate_attachment_mode'] == 'explicit_trigger_event_id'
assert collection['policy'] == POLICY
assert collection['independent_matched_track'] is True
assert collection['neural_role'] == 'spp_candidate_gate'
assert collection['normal_policy_private_state_is_not_nn_input'] is True
assert collection['captured_fill_level_is_replay_action_metadata_not_nn_input'] is True
SCRIPT = f'{REPO}/formal_NN_training/experiments/623_offline_lstm_cnn_spp/python/train_and_offline_infer.py'
CONTRACT = f'{REPO}/formal_NN_training/experiments/623_offline_lstm_cnn_spp/data/stream_contract.json'
assert os.path.isfile(SCRIPT), SCRIPT
contract = json.loads(pathlib.Path(CONTRACT).read_text())
assert contract['experiment_revision'] == 'spp_gate_sliding_cnn_v1'
assert contract['event_logger_schema'] == '623_causal_trigger_v5'
assert contract['causal_candidate_attachment']['mode'] == 'explicit_trigger_event_id'
assert contract['cnn']['temporal_convolution_layers'] == 1
assert contract['cnn']['kernel_size_events'] == 3
assert contract['cnn']['stride_events'] == 1
print(json.dumps(contract['architecture_pairs'], indent=2))


In [ ]:
# Train on local Colab disk; copy only durable outputs to Drive.
LOCAL_INPUT = f'/content/{RUN_ID}_colab_input'
LOCAL_OUTPUT = f'/content/{RUN_ID}_colab_output'
for path in (LOCAL_INPUT, LOCAL_OUTPUT):
    if os.path.isdir(path): shutil.rmtree(path)
shutil.copytree(INPUT_DIR, LOCAL_INPUT)
MODEL_SPECS = [
    {'suffix': 'lstm_h4',  'family': 'lstm', 'size': 4,  'pair_id': 'p0k2'},
    {'suffix': 'cnn_c8',   'family': 'cnn',  'size': 8,  'pair_id': 'p0k2'},
    {'suffix': 'lstm_h8',  'family': 'lstm', 'size': 8,  'pair_id': 'p0k6'},
    {'suffix': 'cnn_c16',  'family': 'cnn',  'size': 16, 'pair_id': 'p0k6'},
    {'suffix': 'lstm_h15', 'family': 'lstm', 'size': 15, 'pair_id': 'p1k6'},
    {'suffix': 'cnn_c32',  'family': 'cnn',  'size': 32, 'pair_id': 'p1k6'},
]
SWEEP = []
local_inputs = {
    role: {
        kind: f'{LOCAL_INPUT}/{TRACE}.{POLICY}.{role}_{kind}.csv.gz'
        for kind in ('stream', 'candidates')
    } for role in ROLES
}
for spec in MODEL_SPECS:
    tag = f"{POLICY}_{spec['suffix']}"
    local_out = f'{LOCAL_OUTPUT}/{tag}'
    drive_out = f'{OUTPUT_ROOT}/{tag}'
    if os.path.isdir(local_out): shutil.rmtree(local_out)
    if os.path.isdir(drive_out): shutil.rmtree(drive_out)
    cmd = [sys.executable, SCRIPT, '--policy', POLICY]
    for role in ROLES:
        cmd += [f'--{role}-stream', local_inputs[role]['stream']]
        cmd += [f'--{role}-candidates', local_inputs[role]['candidates']]
    cmd += [
        '--out-dir', local_out,
        '--model-family', spec['family'],
        '--model-size', str(spec['size']),
        '--pair-id', spec['pair_id'],
        '--device', 'cuda', '--seed', '7', '--epochs', '8',
        '--chunk-len', '1024', '--accumulate-chunks', '16',
    ]
    print('\nTraining', tag, 'command:', ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout: print(result.stdout, end='')
    if result.returncode != 0:
        if result.stderr: print(result.stderr, end='')
        raise RuntimeError(f'{tag} failed with exit code {result.returncode}')
    metadata = json.loads(pathlib.Path(f'{local_out}/run_metadata.json').read_text())
    assert metadata['model_tag'] == tag
    assert metadata['matched_normal_prefetcher'] == POLICY
    assert metadata['neural_role'] == 'spp_candidate_gate'
    assert metadata['model_does_not_use_pc'] is True
    assert metadata['normal_policy_private_state_is_not_nn_input'] is True
    assert metadata['captured_fill_level_is_replay_action_metadata_not_nn_input'] is True
    assert metadata['replay_preserves_captured_fill_level'] is True
    assert metadata['causal_no_future_self_test'] == 'PASS'
    assert metadata['cnn_architecture_self_test'] == 'PASS'
    assert metadata['candidate_rank_normalization'] == 'min(candidate_rank, 32) / 32; fixed before data collection'
    assert metadata['event_logger_schema'] == '623_causal_trigger_v5'
    assert metadata['candidate_attachment_mode'] == 'explicit_trigger_event_id'
    assert metadata['experiment_revision'] == 'spp_gate_sliding_cnn_v1'
    if spec['family'] == 'cnn':
        assert metadata['cnn_temporal_layers'] == 1
        assert metadata['cnn_kernel_size'] == 3
        assert metadata['cnn_stride'] == 1
        assert metadata['cnn_dilation'] == 1
    shutil.copytree(local_out, drive_out)
    SWEEP.append({key: metadata[key] for key in (
        'model_tag', 'matched_normal_prefetcher', 'model_family',
        'model_size', 'architecture_pair_id', 'parameter_count',
        'threshold', 'offline_normal_entries', 'offline_nn_entries',
        'offline_normal_fill_level_counts', 'offline_nn_fill_level_counts'
    )})
manifest = {'trace': TRACE, 'revision': 'spp_gate_sliding_cnn_v1', 'event_logger_schema': '623_causal_trigger_v5', 'candidate_attachment_mode': 'explicit_trigger_event_id', 'points': SWEEP}
pathlib.Path(f'{LOCAL_OUTPUT}/sweep_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
shutil.copy2(f'{LOCAL_OUTPUT}/sweep_manifest.json', f'{OUTPUT_ROOT}/sweep_manifest.json')
print(json.dumps(SWEEP, indent=2))


In [ ]:
for point in SWEEP:
    tag = point['model_tag']
    policy = point['matched_normal_prefetcher']
    required = [f'offline_{policy}.replay.csv', 'offline_nn.replay.csv', 'model.pt', 'run_metadata.json', 'policy_sweep.csv']
    out_dir = f'{OUTPUT_ROOT}/{tag}'
    assert all(os.path.isfile(f'{out_dir}/{name}') for name in required), out_dir
OUTPUT_ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
LOCAL_ARCHIVE = f'/content/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(LOCAL_ARCHIVE, 'w:gz') as archive:
    for item in pathlib.Path(LOCAL_OUTPUT).iterdir():
        archive.add(item, arcname=item.name)
shutil.copy2(LOCAL_ARCHIVE, OUTPUT_ARCHIVE)
print('DONE:', OUTPUT_ARCHIVE, os.path.getsize(OUTPUT_ARCHIVE), 'bytes')


After copying the output archive to the server, run the SPP directory's `replay` stage. The analyzer compares every neural point only with fill-preserving `offline_spp` and reports IPC, L2 miss rate, selected accuracy, coverage, timeliness, and balanced parity.
